# Load Delta Table

In [0]:
df = spark.read.table("workspace.default.facilities2")

# Handle null values

In [0]:
from pyspark.sql.functions import col, when
from pyspark.sql.types import DoubleType, LongType, IntegerType

# Identify column types
double_cols  = [f.name for f in df.schema.fields if isinstance(f.dataType, DoubleType)]
integer_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, (LongType, IntegerType))]
string_cols  = [f.name for f in df.schema.fields if f.name not in double_cols + integer_cols]

# Fill nulls by type — NEVER fill numeric cols with ""
df = df.fillna("", subset=string_cols)   # strings → empty string
df = df.fillna(0.0, subset=double_cols)  # lat/lon → 0.0  (stays as DOUBLE)

# Mark unknown for numberDoctors & capacity columns

In [0]:
df = df.withColumn(
    "numberDoctors",
    when(col("numberDoctors") == 0, "unknown")
    .otherwise(col("numberDoctors").cast("string"))
).withColumn(
    "capacity",
    when(col("capacity") == 0, "unknown")
    .otherwise(col("capacity").cast("string"))
)


# Fix formatting

In [0]:
# Convert to clean bullet text
import json
import re
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType

def parse_list_string(value):
    """
    Parses Python list strings stored as strings in Delta Table.
    Handles:
      - JSON arrays:         '["item1", "item2"]'
      - Python list format:  "['item1', 'item2']"
      - Apostrophes in text: "['GhanaBusinessWeb\\'s page']"
    """
    if not value or str(value).strip() in ("", "[]", "None", "null"):
        return []

    text = str(value).strip()

    # Try JSON first (double quotes) — cleanest parse
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return [str(i).strip() for i in parsed if i and str(i).strip()]
    except (json.JSONDecodeError, ValueError):
        pass

    # Replace ONLY quote chars that are used as list delimiters,
    # NOT apostrophes inside words (e.g. GhanaBusinessWeb's)
    # Strategy: only replace ' that is preceded by [ or , or space
    # and followed by a word character — i.e. opening quotes
    # and ' that is followed by ] or , — i.e. closing quotes
    try:
        # Replace opening list quotes: [' or , ' → [" or , "
        fixed = re.sub(r"(?<=[\[,])\s*'", '"', text)
        # Replace closing list quotes: ', or '] → ", or "]
        fixed = re.sub(r"'\s*(?=[,\]])", '"', fixed)
        parsed = json.loads(fixed)
        if isinstance(parsed, list):
            return [str(i).strip() for i in parsed if i and str(i).strip()]
    except (json.JSONDecodeError, ValueError):
        pass

    # Last resort — use regex to extract quoted strings properly
    # This correctly handles commas INSIDE quoted strings
    try:
        # Match anything between quotes (single or double)
        items = re.findall(r"""['"]([^'"]+)['"]""", text)
        return [i.strip() for i in items if i.strip()]
    except Exception:
        return []

parse_list_udf = udf(parse_list_string, ArrayType(StringType()))

df = df.withColumn("specialties", parse_list_udf(col("specialties"))) \
       .withColumn("procedure",   parse_list_udf(col("procedure")))   \
       .withColumn("equipment",   parse_list_udf(col("equipment")))   \
       .withColumn("capability",  parse_list_udf(col("capability")))

# Remove null / empty values
from pyspark.sql.functions import expr

df = df.withColumn("specialties",
        expr("filter(specialties, x -> x is not null and trim(x) != '')")) \
       .withColumn("procedure",
        expr("filter(procedure,   x -> x is not null and trim(x) != '')")) \
       .withColumn("equipment",
        expr("filter(equipment,   x -> x is not null and trim(x) != '')")) \
       .withColumn("capability",
        expr("filter(capability,  x -> x is not null and trim(x) != '')"))

# Convert to bullet text
df = df.withColumn("specialties_text",
        when(size(col("specialties")) > 0, concat_ws("\n- ", col("specialties")))
        .otherwise("")) \
       .withColumn("capability_text",
        when(size(col("capability")) > 0, concat_ws("\n- ", col("capability")))
        .otherwise("")) \
       .withColumn("procedure_text",
        when(size(col("procedure")) > 0, concat_ws("\n- ", col("procedure")))
        .otherwise("")) \
       .withColumn("equipment_text",
        when(size(col("equipment")) > 0, concat_ws("\n- ", col("equipment")))
        .otherwise(""))


# Remove Noisy data from capability

In [0]:
import re
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType

NOISE_PATTERNS = [
    # Location / address
    "located", "location", "opposite",
    "next to", "near ", "situated", "found in",
    "address:", "address", "based in", "headquaters"
    "country", "city"
    # Contact info
    "telephone", "phone", "tel:", "tel.", "mobile",
    "contact number", "contact us", "call us",
    "email", "e-mail", "mail us","organization name",
    "organization abbreviation"
    # Web / directory
    "website", "web:", "visit us", "visit our",
    "http", "www.", ".com", ".org", ".gh", ".net",
    "listed as", "related place", "directory",
    "ghanabusinessweb", "businessweb", "page","last updated"
    # Admin / registration
    "managed by", "management:", "run by", "operated by",
    "created on", "created by",
    "established in", "established by",
    "registered with", "registered by", "registration",
    "founded in", "founded by",
    "owned by", "ownership", "affiliated with",
    # Geography noise
    "region,", "district,", "municipal", "ghana,",
    # Social media
    "facebook", "twitter", "instagram", "linkedin",
    "follow us", "like us", "likes", "followers"
    # Pricing / hours
    "open from", "opening hours", "hours:", "closed on",
    "consultation fee", "fees:", "charges:",
]

def is_medical_content(text: str) -> bool:
    if not text:
        return False
    stripped = text.strip()
    if len(stripped) < 8:
        return False
    if re.match(r'^[\d\s\-\+\(\)]+$', stripped):
        return False
    text_lower = stripped.lower()
    if any(pattern in text_lower for pattern in NOISE_PATTERNS):
        return False
    return True

def filter_noise_array(items):
    """Filter noise from a parsed array. Input is already a clean Python list."""
    if not items:
        return []
    return [i for i in items if i and is_medical_content(i)]

filter_noise_udf = udf(filter_noise_array, ArrayType(StringType()))

df = df.withColumn("capability", filter_noise_udf(col("capability")))
df = df.withColumn("capability_text",
        when(size(col("capability")) > 0, concat_ws("\n- ", col("capability")))
        .otherwise(""))

# Build Document

In [0]:
from pyspark.sql.functions import concat, lit

df_docs = df.withColumn(
    "document",
    concat(
        lit("[FACILITY]\n"),
        lit("Organization Type: "), col("organization_type"), lit("\n"),
        lit("Name: "), col("name"), lit("\n"),
        lit("Facility Type: "), col("facilityTypeId"), lit("\n\n"),

        lit("[LOCATION]\n"),
        lit("Location: "), col("osm_display_name"), lit("\n\n"),

        lit("[RESOURCES]\n"),
        lit("Number of Doctors: "), col("numberDoctors"), lit("\n"),
        lit("Patient Bed Capacity: "), col("capacity"), lit("\n\n"),

        lit("[MEDICAL DATA]\n"),
        lit("Specialties:\n- "), col("specialties_text"), lit("\n\n"),
        lit("Capabilities:\n- "), col("capability_text"), lit("\n\n"),
        lit("Procedures:\n- "), col("procedure_text"), lit("\n\n"),
        lit("Equipment:\n- "), col("equipment_text"), lit("\n\n"),

        lit("[DESCRIPTION]\n"),
        col("description")
    )
)

# Add metadata as columns

In [0]:
df_docs = df_docs.select(
    col("pk_unique_id").alias("id"),
    "document",
    "name",
    "organization_type",
    col("facilityTypeId").alias("facility_type"),
    "address_city",
    "address_stateOrRegion",
    "address_country",
    "lat",
    "lon",
    "numberDoctors",
    "capacity"
)

# Save delta table

In [0]:
df_docs.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("facility_documents")

# Preview

In [0]:
display(df_docs.select("name", "document").limit(20))